[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_demo09.ipynb)

# MATH840 — Week 9 practice: when one model is not enough

**This session is not marked.** Four cases worked with the tools of the *Advanced Forecasting Methods*
lecture — several seasons, Prophet, weekly data, combinations and their intervals — and every new model
is put next to the ones you already know: seasonal naive, ETS, ARIMA, Theta. Tonight's **Lab 8** is case
4 on your own series.

| Case | Story | What it teaches | Minutes |
|---|---|---|---|
| 0 | — | the toolkit: one backtest for any forecaster | 5 |
| 1 | ⚡ A year of daily electricity in Victoria | two seasons and the weather: who needs which | 25 |
| 2 | 🛒 Walmart, 45 stores, week by week | $m = 52.18$: MSTL, harmonics and Prophet against last year's week | 20 |
| 3 | 🏨 Trips to 76 Australian regions | select one model, or average them? | 15 |
| 4 | 🍽️ Australians eating out | the Lab 8 recipe, end to end: combine, then an honest interval | 20 |

Before each result there is a 🔮 — guess first, then run the cell. Every case ends with a question.

## 0. Setup and the toolkit

In [ ]:
!pip install -q statsforecast==2.1.1 great_tables prophet

import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import logging
import warnings
warnings.filterwarnings("ignore")
for name in ("cmdstanpy", "prophet"):
    logging.getLogger(name).setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from great_tables import GT, loc, style
from matplotlib.colors import LinearSegmentedColormap, to_hex
from prophet import Prophet
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, MSTL, SeasonalNaive

plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")
COLOURS = {"actual": "#314f4f", "a": "#e64173", "b": "#20B2AA", "c": "#FFA500", "d": "#6A5ACD"}

Four tools for the whole session. `backtest` is the one that matters: it takes **any** forecaster — a
function from "the origin" to "the next `h` values" — so ETS, a harmonic regression and Prophet are
scored by exactly the same rule.

In [ ]:
def fourier(t, period, K):
    # Week 5's Fourier terms; the period may be any number, 7, 365.25 or 52.18.
    return np.column_stack([f(2 * np.pi * k * t / period) for k in range(1, K + 1) for f in (np.sin, np.cos)])


def backtest(y, forecasters, origins, h, scale, labels=None):
    # MASE of every forecaster from every origin; also returns the forecasts, for the plots.
    forecasts = {name: [np.asarray(f(e)) for e in origins] for name, f in forecasters.items()}
    table = pd.DataFrame({name: [np.mean(np.abs(y[e:e + h] - fc)) / scale for e, fc in zip(origins, fcs)]
                          for name, fcs in forecasts.items()}, index=labels or origins).T
    table["mean"] = table.mean(axis=1)
    return table, forecasts


def winkler(y, lo, hi, alpha=0.2):
    # Width of the interval, plus 2/alpha times every miss (lecture, Week 8).
    return float(np.mean((hi - lo) + 2 / alpha * (np.maximum(lo - y, 0) + np.maximum(y - hi, 0))))


def coverage(y, lo, hi):
    return float(np.mean((y >= lo) & (y <= hi)))


_shade = LinearSegmentedColormap.from_list("shade", ["#bfe0b0", "#ffffff", "#f5b8cb"])


def scoreboard(table, rules=None, decimals=2):
    # Green is better, pink is worse, the best in bold. A rule is "lower", "higher" or a target value.
    df = table.to_frame() if isinstance(table, pd.Series) else table.copy()
    df.index, df.columns = df.index.map(str), df.columns.map(str)
    rules = rules or {c: "lower" for c in df.columns}
    gt = (GT(df.reset_index(names=" "), rowname_col=" ")
          .fmt_number(columns=list(df.select_dtypes("number").columns), decimals=decimals)
          .sub_missing(missing_text=""))
    for c, rule in rules.items():
        v = df[c].astype(float)
        bad = v if rule == "lower" else -v if rule == "higher" else (v - rule).abs()
        lo, hi = bad.min(), bad.max()
        for i, b in enumerate(bad):
            if np.isnan(b):
                continue
            look = [style.fill(color=to_hex(_shade((b - lo) / (hi - lo) if hi > lo else 0.0)))]
            gt = gt.tab_style(style=look + ([style.text(weight="bold")] if b == lo else []),
                              locations=loc.body(columns=c, rows=[i]))
    return gt

## 1. ⚡ A year of daily electricity in Victoria

Last week's heatwave case used this series for three weeks. Now the question is the grid operator's
everyday one: **how much electricity will the state use each day of the next four weeks?** Daily totals,
2012–2014, with the day's maximum temperature and the public-holiday calendar.

In [ ]:
vic = pd.read_csv(f"{BASE}/vic_elec.csv", parse_dates=["ds"])
wide = vic.pivot_table(index="ds", columns="unique_id", values="y")
holiday = vic.query("unique_id == 'Demand'").set_index("ds")["Holiday"].astype(bool)
daily = pd.DataFrame({"demand": wide["Demand"].resample("D").sum() / 2e3,     # half-hourly MW -> GWh per day
                      "temp": wide["Temperature"].resample("D").max(),
                      "holiday": holiday.resample("D").max()})
daily["workday"] = ((daily.index.dayofweek < 5) & ~daily["holiday"]).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), width_ratios=[2.2, 1, 1])
axes[0].plot(daily.index, daily["demand"], color=COLOURS["actual"], linewidth=0.7)
axes[0].set_title("Daily demand, GWh: a weekly rhythm on a yearly wave")
for flag, colour, label in [(1, COLOURS["d"], "workday"), (0, COLOURS["c"], "weekend or holiday")]:
    part = daily[daily["workday"] == flag]
    axes[1].scatter(part["temp"], part["demand"], s=6, color=colour, alpha=0.6, label=label)
axes[1].set_title("Demand against temperature")
axes[1].set_xlabel("maximum temperature, °C")
axes[1].legend(fontsize=8)
profile = daily.groupby(daily.index.dayofweek)["demand"].mean()
axes[2].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], profile, color=COLOURS["b"])
axes[2].set_ylim(profile.min() * 0.9, profile.max() * 1.03)
axes[2].set_title("Average day of the week")
plt.show()

Three things to read off the plots: a **weekly** season (weekends are low), a **yearly** season (winter
heating, summer air-conditioning), and a **U-shaped** link to temperature — cold days and hot days both
raise demand. ETS and seasonal ARIMA can hold only one season; that is the problem of this case.

In [ ]:
y1 = daily["demand"].to_numpy()
temp = daily["temp"].to_numpy()
t1 = np.arange(len(y1))
H1 = 28
origins1 = [len(y1) - H1 * k for k in range(9, 0, -1)]                 # nine four-week windows, April to December 2014
labels1 = [f"{daily.index[e]:%d %b}" for e in origins1]
scale1 = np.mean(np.abs(y1[7:origins1[0]] - y1[:origins1[0] - 7]))    # seasonal naive (week) MAE before the test

X_season = np.column_stack([fourier(t1, 7, 3), fourier(t1, 365.25, 6)])
X_weather = np.column_stack([X_season, temp, temp ** 2, daily["workday"]])


def harmonic(X):
    # Fourier terms for both seasons (plus weather), ARIMA errors without seasonal terms.
    return lambda e: AutoARIMA(season_length=1, d=0, max_p=3, max_q=3).fit(
        y=y1[:e], X=X[:e]).predict(h=H1, X=X[e:e + H1])["mean"]


def prophet1(weather):
    def forecast(e):
        df = pd.DataFrame({"ds": daily.index, "y": y1, "temp": temp, "temp2": temp ** 2,
                           "workday": daily["workday"].to_numpy()})
        m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        if weather:
            for column in ("temp", "temp2", "workday"):
                m.add_regressor(column)
        m.fit(df.iloc[:e])
        return m.predict(df.iloc[e:e + H1])["yhat"].to_numpy()
    return forecast


forecasters1 = {
    "seasonal naive (week)": lambda e: np.tile(y1[e - 7:e], 4),
    "ETS (m = 7)": lambda e: AutoETS(season_length=7).fit(y=y1[:e]).predict(h=H1)["mean"],
    "ARIMA (m = 7)": lambda e: AutoARIMA(season_length=7).fit(y=y1[:e]).predict(h=H1)["mean"],
    "MSTL [7, 365]": lambda e: MSTL(season_length=[7, 365]).fit(y=y1[:e]).predict(h=H1)["mean"],
    "harmonic regression": harmonic(X_season),
    "harmonic + weather": harmonic(X_weather),
    "Prophet": prophet1(False),
    "Prophet + weather": prophet1(True),
}

🔮 **Guess first.** Eight forecasters, nine four-week windows. Which is better: ETS with the weekly season
only, or MSTL with both? Does temperature help — and by how much? (About two minutes to run.)

In [ ]:
scores1, fc1 = backtest(y1, forecasters1, origins1, H1, scale1, labels1)
scores1.loc["harmonic + weather & Prophet + weather"] = np.nan
pair = [np.mean(np.abs(y1[e:e + H1] - (a + b) / 2)) / scale1
        for e, a, b in zip(origins1, fc1["harmonic + weather"], fc1["Prophet + weather"])]
scores1.loc["harmonic + weather & Prophet + weather"] = pair + [np.mean(pair)]
scoreboard(scores1)

In [ ]:
e = origins1[-1]
days = daily.index[e - 21:e + H1]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(days, y1[e - 21:e + H1], color=COLOURS["actual"], linewidth=1.5, label="actual")
for name, colour, line in [("ETS (m = 7)", COLOURS["d"], ":"), ("MSTL [7, 365]", COLOURS["b"], "-"),
                           ("Prophet + weather", COLOURS["a"], "-"), ("harmonic + weather", COLOURS["c"], "--")]:
    ax.plot(daily.index[e:e + H1], fc1[name][-1], line, color=colour, linewidth=1.6, label=name)
for d in daily.index[e:e + H1][daily["holiday"].iloc[e:e + H1].to_numpy()]:
    ax.axvspan(d - pd.Timedelta(hours=12), d + pd.Timedelta(hours=12), color="grey", alpha=0.15)
ax.axvline(daily.index[e], color="grey", linewidth=0.8)
ax.set_ylabel("GWh per day")
ax.set_title("The last window: December 2014 (grey: public holidays)")
ax.legend(loc="lower left", fontsize=8)
plt.show()

**❓ Question.** The weather models use the temperature that **actually happened** — an ex-post forecast
(Week 7). What would their score be in a real forecast made on 4 December, and what would you give the
operator instead of one line?

## 2. 🛒 Walmart, 45 stores, week by week

Week 4's Walmart data again: weekly sales of 45 stores, February 2010 to October 2012, with a flag on the
four holiday weeks the company itself marks — Super Bowl, Labor Day, Thanksgiving, Christmas. Weekly data
is where the lecture's $m = 52.18$ problem lives: a year is not a whole number of weeks.

First the chain's total, then every store.

In [ ]:
walmart = pd.read_csv(f"{BASE}/walmart.csv", parse_dates=["ds"])
total = walmart.groupby("ds").agg(y=("y", "sum"), holiday=("holiday", "max")).reset_index()
total["y"] /= 1e6                                                       # $ million

fig, ax = plt.subplots(figsize=(12, 3.4))
ax.plot(total["ds"], total["y"], color=COLOURS["actual"], linewidth=1)
for d in total.loc[total["holiday"] == 1, "ds"]:
    ax.axvline(d, color=COLOURS["a"], alpha=0.35, linewidth=3)
ax.set_title("All 45 stores, $ million a week (pink: the four flagged holiday weeks)")
plt.show()

Thanksgiving and Christmas dominate the year; Super Bowl and Labor Day are hard to see. Only **two and a
half years** of data: every method that estimates a yearly shape sees it twice.

Models, from the ones you know to the lecture's:

- **seasonal naive, $m = 52$** — the same week last year;
- **ETS** and **ARIMA without a season** — what you get if you give up on $m = 52$;
- **MSTL with $m = 52$** — the season by smoothing;
- **harmonic regression**, $K = 10$ pairs on the true period 52.18, with and without the holiday flag;
- **Prophet** with the flagged weeks as holidays.

In [ ]:
def weekly_forecasters(y, flag, dates, h):
    t = np.arange(len(y))
    X = fourier(t, 52.18, 10)
    Xh = np.column_stack([X, flag])

    def harmonic(Z):
        return lambda e: AutoARIMA(season_length=1, max_p=3, max_q=3).fit(
            y=y[:e], X=Z[:e]).predict(h=h, X=Z[e:e + h])["mean"]

    def prophet(e):
        holidays = pd.DataFrame({"holiday": "flagged week", "ds": dates[flag == 1]})
        m = Prophet(yearly_seasonality=10, weekly_seasonality=False, daily_seasonality=False,
                    holidays=holidays).fit(pd.DataFrame({"ds": dates[:e], "y": y[:e]}))
        return m.predict(pd.DataFrame({"ds": dates[e:e + h]}))["yhat"].to_numpy()

    return {
        "seasonal naive (m = 52)": lambda e: y[e - 52:e - 52 + h],
        "ETS, no season": lambda e: AutoETS(season_length=1).fit(y=y[:e]).predict(h=h)["mean"],
        "ARIMA, no season": lambda e: AutoARIMA(season_length=1).fit(y=y[:e]).predict(h=h)["mean"],
        "MSTL (m = 52)": lambda e: MSTL(season_length=52).fit(y=y[:e]).predict(h=h)["mean"],
        "harmonic, K = 10": harmonic(X),
        "harmonic + holidays": harmonic(Xh),
        "Prophet + holidays": prophet,
    }


y2 = total["y"].to_numpy()
H2 = 13                                                                 # a quarter ahead
origins2 = [len(y2) - H2 * k for k in (3, 2, 1)]
labels2 = [f"{total['ds'][e]:%d %b %Y}" for e in origins2]
scale2 = np.mean(np.abs(y2[52:origins2[0]] - y2[:origins2[0] - 52]))

🔮 **Guess first.** The lecture's weekly electricity had two years of training data, and the same week
last year won. Will it win here too?

In [ ]:
scores2, fc2 = backtest(y2, weekly_forecasters(y2, total["holiday"].to_numpy(), total["ds"].to_numpy(), H2),
                        origins2, H2, scale2, labels2)
scoreboard(scores2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.plot(total["ds"].iloc[52:], y2[52:], color=COLOURS["actual"], linewidth=1.2, label="actual")
for name, colour, line in [("seasonal naive (m = 52)", COLOURS["d"], ":"), ("MSTL (m = 52)", COLOURS["b"], "-"),
                           ("Prophet + holidays", COLOURS["a"], "--")]:
    for i, e in enumerate(origins2):
        ax.plot(total["ds"].iloc[e:e + H2], fc2[name][i], line, color=colour, linewidth=1.6,
                label=name if i == 0 else None)
for e in origins2:
    ax.axvline(total["ds"].iloc[e], color="grey", linewidth=0.8)
ax.set_title("Three quarters of 2012, each forecast from its own origin")
ax.legend(loc="upper left", fontsize=8)
plt.show()

One series is one draw. The same seven forecasters on **every store** — 45 × 3 windows. (About four
minutes; Prophet and the regressions are refitted 135 times each.)

In [ ]:
rows = {}
for store, g in walmart.groupby("unique_id"):
    y, flag, dates = g["y"].to_numpy() / 1e6, g["holiday"].to_numpy(), g["ds"].to_numpy()
    scale = np.mean(np.abs(y[52:origins2[0]] - y[:origins2[0] - 52]))
    table, fcs = backtest(y, weekly_forecasters(y, flag, dates, H2), origins2, H2, scale)
    combo = [np.mean(np.abs(y[e:e + H2] - (a + b) / 2)) / scale
             for e, a, b in zip(origins2, fcs["MSTL (m = 52)"], fcs["Prophet + holidays"])]
    rows[store] = table["mean"].to_dict() | {"MSTL & Prophet": np.mean(combo)}
stores = pd.DataFrame(rows).T

summary2 = pd.DataFrame({"mean MASE": stores.mean(), "median MASE": stores.median(),
                         "beats last year's week": stores.lt(stores["seasonal naive (m = 52)"], axis=0).mean(),
                         "best of the seven": stores.drop(columns="MSTL & Prophet").idxmin(axis=1)
                         .value_counts().reindex(stores.columns).fillna(0) / len(stores)})
scoreboard(summary2, rules={"mean MASE": "lower", "median MASE": "lower",
                            "beats last year's week": "higher", "best of the seven": "higher"})

**❓ Question.** The holiday flag barely changes the harmonic regression or Prophet. Look at the dates of the
three test quarters: which flagged holidays fall inside them? What would the flag be worth from an origin
in early November?

## 3. 🏨 Trips to 76 Australian regions: select or combine?

The lecture ran this experiment on the Track B pool, which you cannot see. Here it is on public data:
quarterly overnight trips to each of Australia's 76 tourism regions, 1998–2017, all purposes together.

For every region:

- **select** — score ETS, ARIMA and Theta on three validation windows, keep the best, use it on the test;
- **combine** — average the forecasts, no choice made.

Test: the last two years (8 quarters), never seen by either.

In [ ]:
tourism = pd.read_csv(f"{BASE}/tourism.csv", parse_dates=["ds"])
regions = tourism.groupby(["Region", "ds"], as_index=False)["y"].sum().rename(columns={"Region": "unique_id"})
H3 = 8
test3 = regions.groupby("unique_id").tail(H3)
train3 = regions.drop(test3.index)
names3 = ["ETS", "ARIMA", "Theta"]
sf3 = StatsForecast(models=[AutoETS(season_length=4, alias="ETS"), AutoARIMA(season_length=4, alias="ARIMA"),
                            AutoTheta(season_length=4, alias="Theta"), SeasonalNaive(4, alias="seasonal naive")],
                    freq="QS", n_jobs=-1)
validation = sf3.cross_validation(df=train3, h=H3, n_windows=3, step_size=4)      # three origins, a year apart
tested = sf3.forecast(df=train3, h=H3).merge(test3, on=["unique_id", "ds"])
scale3 = train3.groupby("unique_id")["y"].apply(lambda s: np.mean(np.abs(s.to_numpy()[4:] - s.to_numpy()[:-4])))


def mase_by_series(df, columns):
    return df.groupby("unique_id").apply(
        lambda g: pd.Series({c: np.mean(np.abs(g["y"] - g[c])) for c in columns}), include_groups=False
    ).div(scale3, axis=0)


tested["average of three"] = tested[names3].mean(axis=1)
val3 = mase_by_series(validation, names3)
test_scores3 = mase_by_series(tested, names3 + ["seasonal naive", "average of three"])
picked = val3.idxmin(axis=1)
test_scores3["select on validation"] = [test_scores3.loc[r, picked[r]] for r in test_scores3.index]
test_scores3["oracle: the test winner"] = test_scores3[names3].min(axis=1)
print("selection picked:", picked.value_counts().to_dict())

🔮 **Guess first.** Three candidates: a coin would pick the test winner a third of the time. How often does
selection on three validation windows pick it?

In [ ]:
hit = (picked == test_scores3[names3].idxmin(axis=1)).mean()
print(f"selection picked the model that won the test in {hit:.0%} of regions (a coin: 33%)")
summary3 = pd.DataFrame({"mean MASE": test_scores3.mean(), "median MASE": test_scores3.median(),
                         "worst region": test_scores3.max(),
                         "beats seasonal naive": test_scores3.lt(test_scores3["seasonal naive"], axis=0).mean()})
scoreboard(summary3, rules={"mean MASE": "lower", "median MASE": "lower", "worst region": "lower",
                            "beats seasonal naive": "higher"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
a, b = test_scores3["select on validation"], test_scores3["average of three"]
axes[0].scatter(a, b, s=18, color=np.where(b < a, COLOURS["b"], COLOURS["a"]))
lim = [min(a.min(), b.min()) * 0.95, max(a.max(), b.max()) * 1.05]
axes[0].plot(lim, lim, color="grey", linewidth=0.8)
axes[0].set_xlabel("MASE, select on validation")
axes[0].set_ylabel("MASE, average of three")
axes[0].set_title(f"A dot per region; below the line the average wins ({(b < a).mean():.0%})")
gap = (b - a).sort_values()
axes[1].bar(range(len(gap)), gap, color=np.where(gap < 0, COLOURS["b"], COLOURS["a"]))
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Average minus selection, sorted")
axes[1].set_xlabel("region")
plt.show()

**❓ Question.** The oracle row is the best you could do if you always knew the winner. How much of the gap
between "select" and the oracle does the average close? Why is selection on three windows barely better
than — or worse than — a coin?

## 4. 🍽️ Australians eating out: the Lab 8 recipe, end to end

Monthly turnover of Australia's cafés, restaurants and takeaway food services, all states together,
1982–2018. The national total, not a state: no Track B series is a national total.

Tonight's Lab 8, step by step:

1. **log**, since the seasonal swing grows with the level;
2. **validation** — ETS, ARIMA and MSTL on six rolling 12-month windows before the test;
3. **combination** — the plain average of the three;
4. **interval** — three recipes, scored by coverage and the Winkler score;
5. **forecast** — refit on everything, hand in.

In [ ]:
retail = pd.read_csv(f"{BASE}/aus_retail.csv", parse_dates=["Month"])
national = (retail.groupby(["Industry", "Month"], as_index=False)["Turnover"].sum()
            .rename(columns={"Industry": "unique_id", "Month": "ds", "Turnover": "y"}))
cafes = national.query("unique_id == 'Cafes, restaurants and takeaway food services'").reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(cafes["ds"], cafes["y"], color=COLOURS["actual"])
axes[0].set_title("Turnover, $ million a month")
axes[1].plot(cafes["ds"], np.log(cafes["y"]), color=COLOURS["b"])
axes[1].set_title("log(turnover): the December peak no longer grows")
plt.show()

In [ ]:
H4, WINDOWS = 12, 6
MEMBERS = ["ETS", "ARIMA", "MSTL"]


def members():
    return [AutoETS(season_length=12, alias="ETS"), AutoARIMA(season_length=12, alias="ARIMA"),
            MSTL(season_length=12, alias="MSTL")]


logged = cafes.assign(y=np.log(cafes["y"]))
train4, test4 = logged.iloc[:-H4], cafes.iloc[-H4:]
sf4 = StatsForecast(models=members(), freq="MS")
cv4 = sf4.cross_validation(df=train4, h=H4, n_windows=WINDOWS, step_size=H4)
cv4["combination"] = cv4[MEMBERS].mean(axis=1)
past4 = cafes["y"].iloc[:-H4].to_numpy()
scale4 = np.mean(np.abs(past4[12:] - past4[:-12]))                     # seasonal naive MAE, $ million
val4 = cv4.groupby("cutoff").apply(
    lambda g: pd.Series({c: np.mean(np.abs(np.exp(g["y"]) - np.exp(g[c]))) / scale4 for c in MEMBERS + ["combination"]}),
    include_groups=False).T
val4.columns = [f"from {c:%b %Y}" for c in val4.columns]
val4["mean"] = val4.mean(axis=1)

🔮 **Guess first.** Six validation windows. Will one member win all six? Where will the combination land?

In [ ]:
scoreboard(val4)

Now the test year — 2018, never touched. The members are refitted on everything before it. Selection takes
the member with the best validation mean; the combination takes no decision.

Three intervals for the combination:

- **each member's own** 80% interval, for comparison;
- **averaged bounds** — the mean of the members' lower and upper bounds;
- **conformal** — the combination's errors on the six validation windows, **one set per horizon step**
  (6 errors for $h = 1$, 6 for $h = 2$, …), and their 10% and 90% quantiles added to the new forecast. The
  lecture's conformal interval used one window and failed; this one uses six.

In [ ]:
fc4 = sf4.forecast(df=train4, h=H4, level=[80])
fc4["combination"] = fc4[MEMBERS].mean(axis=1)
actual4 = test4["y"].to_numpy()
best_member = val4.loc[MEMBERS, "mean"].idxmin()

points = {m: np.exp(fc4[m].to_numpy()) for m in MEMBERS}
points["combination"] = np.exp(fc4["combination"].to_numpy())
points[f"select on validation ({best_member})"] = points[best_member]
point_scores = pd.Series({k: np.mean(np.abs(actual4 - v)) / scale4 for k, v in points.items()}, name="test MASE")

cv4["step"] = cv4.groupby("cutoff").cumcount()                         # 0 .. 11: the horizon step
errors = (cv4["y"] - cv4["combination"]).groupby(cv4["step"])          # log errors, six per step
q_lo, q_hi = errors.quantile(0.10).to_numpy(), errors.quantile(0.90).to_numpy()

intervals = {f"{m}'s own": (np.exp(fc4[f"{m}-lo-80"]), np.exp(fc4[f"{m}-hi-80"])) for m in MEMBERS}
intervals["averaged bounds"] = (np.exp(fc4[[f"{m}-lo-80" for m in MEMBERS]].mean(axis=1)),
                                np.exp(fc4[[f"{m}-hi-80" for m in MEMBERS]].mean(axis=1)))
intervals["conformal, six windows"] = (np.exp(fc4["combination"] + q_lo), np.exp(fc4["combination"] + q_hi))
interval_scores = pd.DataFrame({k: {"coverage": coverage(actual4, np.asarray(lo), np.asarray(hi)),
                                    "Winkler": winkler(actual4, np.asarray(lo), np.asarray(hi)),
                                    "mean width": float(np.mean(np.asarray(hi) - np.asarray(lo)))}
                                for k, (lo, hi) in intervals.items()}).T

🔮 **Guess first.** Did selection pick the right member for 2018? Which interval is closest to 80%
coverage — and is the narrowest one the best by Winkler?

In [ ]:
display(scoreboard(point_scores))
scoreboard(interval_scores, rules={"coverage": 0.8, "Winkler": "lower", "mean width": "lower"})

In [ ]:
months = test4["ds"]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(cafes["ds"].iloc[-48:], cafes["y"].iloc[-48:], color=COLOURS["actual"], linewidth=1.3, label="actual")
ax.plot(months, points["combination"], color=COLOURS["a"], linewidth=2, label="combination")
lo, hi = intervals["averaged bounds"]
ax.fill_between(months, lo, hi, color=COLOURS["b"], alpha=0.2, label="averaged bounds")
lo, hi = intervals["conformal, six windows"]
ax.fill_between(months, lo, hi, color=COLOURS["c"], alpha=0.35, label="conformal, six windows")
ax.axvline(months.iloc[0], color="grey", linewidth=0.8)
ax.set_title("2018: the combination and its two intervals")
ax.legend(loc="upper left", fontsize=8)
plt.show()

### Is one series enough? The recipe as a function, on all 20 industries

One series is one story. The same five steps, packed into a function, on every national retail industry
— and that function is your Lab 8. (About two minutes.)

In [ ]:
def lab8_recipe(df, h=12, windows=6):
    # Steps 1-4 for every series in df (columns unique_id, ds, y > 0); returns one row per series.
    logged = df.assign(y=np.log(df["y"]))
    test = df.groupby("unique_id").tail(h)
    train = logged.drop(test.index)
    sf = StatsForecast(models=members(), freq="MS", n_jobs=-1)
    cv = sf.cross_validation(df=train, h=h, n_windows=windows, step_size=h)
    cv["combination"] = cv[MEMBERS].mean(axis=1)
    cv["step"] = cv.groupby(["unique_id", "cutoff"]).cumcount()
    fc = sf.forecast(df=train, h=h, level=[80]).merge(test, on=["unique_id", "ds"])
    fc["combination"] = fc[MEMBERS].mean(axis=1)
    fc["step"] = fc.groupby("unique_id").cumcount()
    rows = {}
    for uid, g in fc.groupby("unique_id"):
        past = df.loc[df["unique_id"] == uid, "y"].to_numpy()[:-h]
        scale = np.mean(np.abs(past[12:] - past[:-12]))
        v = cv[cv["unique_id"] == uid]
        val = {m: np.mean(np.abs(np.exp(v["y"]) - np.exp(v[m]))) for m in MEMBERS}
        pick = min(val, key=val.get)
        err = (v["y"] - v["combination"]).groupby(v["step"])
        lo = np.exp(g["combination"].to_numpy() + err.quantile(0.10).to_numpy())
        hi = np.exp(g["combination"].to_numpy() + err.quantile(0.90).to_numpy())
        alo = np.exp(g[[f"{m}-lo-80" for m in MEMBERS]].mean(axis=1).to_numpy())
        ahi = np.exp(g[[f"{m}-hi-80" for m in MEMBERS]].mean(axis=1).to_numpy())
        y, mase = g["y"].to_numpy(), lambda f: np.mean(np.abs(g["y"] - np.exp(f))) / scale
        rows[uid] = {**{m: mase(g[m]) for m in MEMBERS}, "combination": mase(g["combination"]),
                     "select on validation": mase(g[pick]), "picked": pick,
                     "coverage, averaged": coverage(y, alo, ahi), "coverage, conformal": coverage(y, lo, hi),
                     "Winkler, averaged": winkler(y, alo, ahi) / scale, "Winkler, conformal": winkler(y, lo, hi) / scale}
    return pd.DataFrame(rows).T


industries = lab8_recipe(national)
numeric = industries.drop(columns="picked").astype(float)
summary4 = pd.DataFrame({"mean": numeric.mean(), "median": numeric.median()})
print("combination beats selection on", f"{(numeric['combination'] < numeric['select on validation']).mean():.0%}",
      "of industries; selection picked:", industries["picked"].value_counts().to_dict())
scoreboard(summary4.T, rules={"ETS": "lower", "ARIMA": "lower", "MSTL": "lower", "combination": "lower",
                              "select on validation": "lower", "coverage, averaged": 0.8,
                              "coverage, conformal": 0.8, "Winkler, averaged": "lower",
                              "Winkler, conformal": "lower"})

**❓ Question.** Your Lab 8 series is one of these rows. Which of tonight's decisions would you take
before seeing your validation table — log or not, which members, combine or select, which interval — and
which ones should the table decide?

## What carries to tonight

| Case | What we compared | What decided it |
|---|---|---|
| ⚡ Electricity | one season against two, with and without weather | two seasons beat one; weather beat both — ex post |
| 🛒 Walmart | last year's week against MSTL, harmonics, Prophet | with 2.5 years, MSTL — not last year's week — on most stores |
| 🏨 Tourism | select on validation against the average | selection is close to a coin; the average costs nothing |
| 🍽️ Eating out | three members, three intervals | conformal from six windows is the calibrated one |

**Lab 8** is case 4 on your own Track B series: log if the swing grows, ETS + ARIMA + MSTL validated on
rolling windows, the combination as the forecast, a conformal interval from the validation errors —
scored, explained, handed in.